In [0]:
SUPABASE_URL = ""

SUPABASE_KEY = ""

In [0]:
from supabase import create_client

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_KEY
)

In [0]:
%pip install supabase

In [0]:
dbutils.library.restartPython()

In [0]:
# config
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql.functions import (
    col, count, sum as sql_sum, avg,
    min as sql_min, max as sql_max,
    year, month, quarter, dayofweek,
    datediff, when, desc, asc, round
)

from pyspark.sql.window import Window

from supabase import create_client, Client


#plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [0]:
pip install supabase python-dotenv

In [0]:
#carregando dados
df = spark.table("fiap.silver.SINAN_DENGUE_SP")

print(f"\n Dados carregados:")
print(f"total de registros: {df.count():,.0f}")
print(f"período: {df.select(sql_min('DATA_NOTIFICACAO')).collect()[0][0]} a {df.select(sql_max('DATA_NOTIFICACAO')).collect()[0][0]}")
print(f"total de colunas: {len(df.columns)}")

#esquema
print(f"\n colunas e dados:")
df.printSchema()

In [0]:
from decimal import Decimal

def enviar_para_supabase(
    tabela_spark,
    nome_tabela_supabase,
    truncate=True
):

    try:
        df_pandas = tabela_spark.toPandas()

       #converter coluna date time para string
        for col in df_pandas.columns:

            if pd.api.types.is_datetime64_any_dtype(df_pandas[col]):
                df_pandas[col] = df_pandas[col].astype(str)

            # objetos date do python
            df_pandas[col] = df_pandas[col].apply(
                lambda x: str(x)
                if hasattr(x, 'isoformat')
                else x
            )

        # decimal pra float
        for col in df_pandas.columns:
            df_pandas[col] = df_pandas[col].apply(
                lambda x: float(x)
                if isinstance(x, Decimal)
                else x
            )

        #nan pra none
        df_pandas = df_pandas.where(
            pd.notna(df_pandas),
            None
        )

        #converter nomes das colunas para lowercase
        df_pandas.columns = [
            col.lower()
            for col in df_pandas.columns
        ]

        #lista de dicts
        dados = df_pandas.to_dict('records')

        print(f"\nenviando {nome_tabela_supabase}")
        print(f"total {len(dados):,.0f} registros")

        
        # limpar tabela se solicitado
        if truncate:
            try:

                #pega primeira coluna da tabela
                primeira_coluna = df_pandas.columns[0]

                #delete all
                supabase.table(nome_tabela_supabase)\
                    .delete()\
                    .not_.is_(primeira_coluna, "null")\
                    .execute()

                print(f"tabela {nome_tabela_supabase} limpa")

            except Exception as e:
                print(f"não foi possivel limpar: {e}")

        #inserindo dados em chunks
        chunk_size = 1000
        total_inserido = 0

        for i in range(0, len(dados), chunk_size):

            chunk = dados[i:i+chunk_size]

            supabase.table(nome_tabela_supabase)\
                .insert(chunk)\
                .execute()

            total_inserido += len(chunk)
            
            print(
                f"inseridos {len(chunk):,.0f} registros "
                f"({total_inserido:,.0f}/{len(dados):,.0f})"
            )

        print(f"{nome_tabela_supabase} enviado com sucesso\n")

        return True

    except Exception as e:
        print(f"erro ao enviar {nome_tabela_supabase}: {e}\n")
        return False

In [0]:
#analises

#distribuição por sexo
df_sexo = spark.sql("""
SELECT 
    GENERO,
    COUNT(*) AS total_casos,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY GENERO
ORDER BY total_casos DESC
""")
enviar_para_supabase(df_sexo, "analise_sexo")

# distribuição por faixa etária
print("\ndistribuição por faixa etaria:")
df_faixa = spark.sql("""
SELECT 
    CASE 
        WHEN IDADE_ANOS < 5 THEN '0-4 anos'
        WHEN IDADE_ANOS < 15 THEN '5-14 anos'
        WHEN IDADE_ANOS < 30 THEN '15-29 anos'
        WHEN IDADE_ANOS < 45 THEN '30-44 anos'
        WHEN IDADE_ANOS < 60 THEN '45-59 anos'
        WHEN IDADE_ANOS >= 60 THEN '60+ anos'
        ELSE 'Desconhecido'
    END AS faixa_etaria,
    COUNT(*) AS total_casos,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct,
    ROUND(AVG(CASE WHEN FLAG_HOSPITALIZACAO = 1 THEN 1 ELSE 0 END) * 100, 2) AS taxa_hosp_pct,
    ROUND(AVG(CASE WHEN FLAG_OBITO_DENGUE = 1 THEN 1 ELSE 0 END) * 100, 2) AS taxa_mortalidade_pct
FROM fiap.silver.SINAN_DENGUE_SP
WHERE IDADE_ANOS IS NOT NULL
GROUP BY faixa_etaria
ORDER BY 
    CASE 
        WHEN faixa_etaria = '0-4 anos' THEN 1
        WHEN faixa_etaria = '5-14 anos' THEN 2
        WHEN faixa_etaria = '15-29 anos' THEN 3
        WHEN faixa_etaria = '30-44 anos' THEN 4
        WHEN faixa_etaria = '45-59 anos' THEN 5
        WHEN faixa_etaria = '60+ anos' THEN 6
        ELSE 7
    END
""")
enviar_para_supabase(df_faixa, "analise_faixa_etaria")

# distribuicao por raça/cor
print("\n distribuicao por raça/cor:")
df_raca = spark.sql("""
SELECT 
    RACA_COR,
    COUNT(*) AS total_casos,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY RACA_COR
ORDER BY total_casos DESC
""")
enviar_para_supabase(df_raca, "analise_raca_cor")

# por ano
print("resumo por ano:")
df_ano = spark.sql("""
SELECT 
    ANO_NOTIFIC,
    COUNT(*) AS total_casos,
    COUNT(DISTINCT MONTH(DATA_NOTIFICACAO)) AS meses_com_casos,
    ROUND(AVG(CASE WHEN FLAG_HOSPITALIZACAO = 1 THEN 1 ELSE 0 END) * 100, 2) AS taxa_hosp_pct,
    ROUND(AVG(CASE WHEN FLAG_OBITO_DENGUE = 1 THEN 1 ELSE 0 END) * 100, 2) AS taxa_mortalidade_pct,
    MIN(DATA_NOTIFICACAO) AS primeira_data,
    MAX(DATA_NOTIFICACAO) AS ultima_data
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY ANO_NOTIFIC
ORDER BY ANO_NOTIFIC
""")
enviar_para_supabase(df_ano, "analise_por_ano")

In [0]:
#analises

# resumo
print("\n resumo:")
df_resumo = spark.sql("""
SELECT 
    COUNT(*) AS total_registros,
    SUM(FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(FLAG_OBITO_DENGUE) AS obitos_dengue,
    SUM(FLAG_OBITO_GERAL) AS obitos_gerais,
    ROUND(SUM(FLAG_HOSPITALIZACAO) * 100.0 / COUNT(*), 2) AS taxa_hosp_pct,
    ROUND(SUM(FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 4) AS taxa_morte_dengue_pct,
    ROUND(SUM(FLAG_OBITO_GERAL) * 100.0 / COUNT(*), 4) AS taxa_morte_geral_pct
FROM fiap.silver.SINAN_DENGUE_SP
""")
enviar_para_supabase(df_resumo, "desfecho_resumo_geral")

print("\n desfechos por faixa etaria:")
df_desfecho_faixa = spark.sql("""
SELECT 
    CASE 
        WHEN IDADE_ANOS < 5 THEN '0-4 anos'
        WHEN IDADE_ANOS < 15 THEN '5-14 anos'
        WHEN IDADE_ANOS < 30 THEN '15-29 anos'
        WHEN IDADE_ANOS < 45 THEN '30-44 anos'
        WHEN IDADE_ANOS < 60 THEN '45-59 anos'
        WHEN IDADE_ANOS >= 60 THEN '60+ anos'
        ELSE 'Desconhecido'
    END AS faixa_etaria,
    COUNT(*) AS total,
    SUM(FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(FLAG_OBITO_DENGUE) AS obitos,
    ROUND(SUM(FLAG_HOSPITALIZACAO) * 100.0 / COUNT(*), 2) AS taxa_hosp_pct,
    ROUND(SUM(FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 2) AS taxa_morte_pct
FROM fiap.silver.SINAN_DENGUE_SP
WHERE IDADE_ANOS IS NOT NULL
GROUP BY faixa_etaria
ORDER BY 
    CASE 
        WHEN faixa_etaria = '0-4 anos' THEN 1
        WHEN faixa_etaria = '5-14 anos' THEN 2
        WHEN faixa_etaria = '15-29 anos' THEN 3
        WHEN faixa_etaria = '30-44 anos' THEN 4
        WHEN faixa_etaria = '45-59 anos' THEN 5
        WHEN faixa_etaria = '60+ anos' THEN 6
        ELSE 7
    END
""")
enviar_para_supabase(df_desfecho_faixa, "desfecho_por_faixa_etaria")

# desfechos por sexo
print("\n desfechos por sexo:")
df_desfecho_sexo = spark.sql("""
SELECT 
    GENERO,
    COUNT(*) AS total,
    SUM(FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(FLAG_OBITO_DENGUE) AS obitos,
    ROUND(SUM(FLAG_HOSPITALIZACAO) * 100.0 / COUNT(*), 2) AS taxa_hosp_pct,
    ROUND(SUM(FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 2) AS taxa_morte_pct
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY GENERO
ORDER BY total DESC
""")
enviar_para_supabase(df_desfecho_sexo, "desfecho_por_sexo")

#por ano
print("\n desfechos por ano:")
df_desfecho_ano = spark.sql("""
SELECT 
    ANO_NOTIFIC,
    COUNT(*) AS total,
    SUM(FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(FLAG_OBITO_DENGUE) AS obitos,
    ROUND(SUM(FLAG_HOSPITALIZACAO) * 100.0 / COUNT(*), 2) AS taxa_hosp_pct,
    ROUND(SUM(FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 2) AS taxa_morte_pct
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY ANO_NOTIFIC
ORDER BY ANO_NOTIFIC
""")
enviar_para_supabase(df_desfecho_ano, "desfecho_por_ano")

In [0]:
#detectar surtos

# casos por mes
print("\n tendecia mensal")
df_tendencia = spark.sql("""
SELECT 
    YEAR(DATA_NOTIFICACAO) AS ano,
    MONTH(DATA_NOTIFICACAO) AS mes,
    COUNT(*) AS casos,
    SUM(FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(FLAG_OBITO_DENGUE) AS obitos,
    ROUND(SUM(FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 2) AS taxa_mortalidade_pct
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY YEAR(DATA_NOTIFICACAO), MONTH(DATA_NOTIFICACAO)
ORDER BY ano, mes
""")
enviar_para_supabase(df_tendencia, "surto_tendencia_mensal")

# calculo linha base e alerta
print("\n Analise de mudanca  (2024):")
df_mudanca = spark.sql("""
SELECT 
    ANO_NOTIFIC,
    COUNT(*) AS total_casos,
    ROUND(AVG(COUNT(*)) OVER (ORDER BY ANO_NOTIFIC ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0) AS media_anos_anteriores,
    ROUND(COUNT(*) / AVG(COUNT(*)) OVER (ORDER BY ANO_NOTIFIC ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 2) AS multiplicador
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY ANO_NOTIFIC
ORDER BY ANO_NOTIFIC
""")
enviar_para_supabase(df_mudanca, "surto_mudanca_regime")

In [0]:
print("Analises geo")


# top 10 municipios com mais casos
print("\n Top 10 municipios com mais casos:")
df_municipios_top = spark.sql("""
SELECT 
    s.COD_IBGE_MUNICIPIO,
    i.NOME_MUNICIPIO,
    i.NOME_MESORREGIAO,
    COUNT(*) AS total_casos,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fiap.silver.SINAN_DENGUE_SP), 2) AS pct_estado,
    SUM(s.FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(s.FLAG_OBITO_DENGUE) AS obitos,
    ROUND(SUM(s.FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 2) AS taxa_mortalidade_pct,
    MIN(s.DATA_NOTIFICACAO) AS primeira_notificacao,
    MAX(s.DATA_NOTIFICACAO) AS ultima_notificacao
FROM fiap.silver.SINAN_DENGUE_SP s
LEFT JOIN fiap.silver.IBGE_SP i ON CAST(s.COD_IBGE_MUNICIPIO AS BIGINT) = CAST(i.COD_SUS AS BIGINT)
GROUP BY s.COD_IBGE_MUNICIPIO, i.NOME_MUNICIPIO, i.NOME_MESORREGIAO
ORDER BY total_casos DESC
LIMIT 10
""")
enviar_para_supabase(df_municipios_top, "geo_top_10_municipios")

#top 10 municipios com mais obitos
print("\nTop 10 municipios com mais óbitos:")
df_municipios_obitos = spark.sql("""
SELECT 
    s.COD_IBGE_MUNICIPIO,
    i.NOME_MUNICIPIO,
    i.NOME_MESORREGIAO,
    COUNT(*) AS total_casos,
    SUM(s.FLAG_OBITO_DENGUE) AS obitos,
    ROUND(SUM(s.FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 2) AS taxa_mortalidade_pct
FROM fiap.silver.SINAN_DENGUE_SP s
LEFT JOIN fiap.silver.IBGE_SP i ON CAST(s.COD_IBGE_MUNICIPIO AS BIGINT) = CAST(i.COD_SUS AS BIGINT)
GROUP BY s.COD_IBGE_MUNICIPIO, i.NOME_MUNICIPIO, i.NOME_MESORREGIAO
HAVING SUM(s.FLAG_OBITO_DENGUE) > 0
ORDER BY obitos DESC
LIMIT 10
""")
enviar_para_supabase(df_municipios_obitos, "geo_top_10_obitos")

#distribuição geografica meso
print("\ndistribuição de casos por mesorregião:")
df_mesorregiao = spark.sql("""
SELECT 
    i.NOME_MESORREGIAO,
    COUNT(DISTINCT s.COD_IBGE_MUNICIPIO) AS municipios,
    COUNT(*) AS total_casos,
    SUM(s.FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(s.FLAG_OBITO_DENGUE) AS obitos,
    ROUND(SUM(s.FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 2) AS taxa_mortalidade_pct,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fiap.silver.SINAN_DENGUE_SP), 2) AS pct_estado
FROM fiap.silver.SINAN_DENGUE_SP s
LEFT JOIN fiap.silver.IBGE_SP i ON CAST(s.COD_IBGE_MUNICIPIO AS BIGINT) = CAST(i.COD_SUS AS BIGINT)
GROUP BY i.NOME_MESORREGIAO
ORDER BY total_casos DESC
""")
enviar_para_supabase(df_mesorregiao, "geo_por_mesorregiao")

#distribuição geografica por ano e mesorregião
print("\nEvolução de casos por mesorregião:")
df_mesorregiao_ano = spark.sql("""
SELECT 
    YEAR(s.DATA_NOTIFICACAO) AS ano,
    i.NOME_MESORREGIAO,
    COUNT(*) AS total_casos,
    SUM(s.FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(s.FLAG_OBITO_DENGUE) AS obitos
FROM fiap.silver.SINAN_DENGUE_SP s
LEFT JOIN fiap.silver.IBGE_SP i ON CAST(s.COD_IBGE_MUNICIPIO AS BIGINT) = CAST(i.COD_SUS AS BIGINT)
GROUP BY ano, i.NOME_MESORREGIAO
ORDER BY ano DESC, total_casos DESC
""")
enviar_para_supabase(df_mesorregiao_ano, "geo_mesorregiao_por_ano")

#número de municipios afetados por ano
print("\n número de municípios afetados por ano:")
df_municipios_ano = spark.sql("""
SELECT 
    YEAR(s.DATA_NOTIFICACAO) AS ano,
    COUNT(DISTINCT s.COD_IBGE_MUNICIPIO) AS municipios_afetados,
    COUNT(*) AS total_casos,
    ROUND(COUNT(*) / COUNT(DISTINCT s.COD_IBGE_MUNICIPIO), 0) AS casos_por_municipio
FROM fiap.silver.SINAN_DENGUE_SP s
GROUP BY ano
ORDER BY ano
""")
enviar_para_supabase(df_municipios_ano, "geo_municipios_por_ano")

In [0]:
#sazonalidade

# padrao sazonal por mes
print("\n padrao sazonal por mes):")
df_sazon = spark.sql("""
SELECT 
    MONTH(DATA_NOTIFICACAO) AS mes,
    COUNT(*) AS total_casos,
    ROUND(AVG(COUNT(*)) OVER (), 0) AS media_geral,
    ROUND(COUNT(*) / AVG(COUNT(*)) OVER () * 100, 1) AS pct_media,
    SUM(FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(FLAG_OBITO_DENGUE) AS obitos
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY MONTH(DATA_NOTIFICACAO)
ORDER BY mes
""")
enviar_para_supabase(df_sazon, "sazonalidade_por_mes")

# sazonalidade por ano
print("\n sazonalidade por ano:")
df_sazon_ano = spark.sql("""
SELECT 
    ANO_NOTIFIC,
    MONTH(DATA_NOTIFICACAO) AS mes,
    COUNT(*) AS casos
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY ANO_NOTIFIC, MONTH(DATA_NOTIFICACAO)
ORDER BY ANO_NOTIFIC, casos DESC
""")
enviar_para_supabase(df_sazon_ano, "sazonalidade_por_ano")

##Gold

In [0]:
#criacao tabelas gold

# tabela por municipio
print("\ncriando tabela SINAN_DENGUE_MUNICIPIOS_DETALHADO")
spark.sql("""
CREATE OR REPLACE TABLE fiap.gold.SINAN_DENGUE_MUNICIPIOS_DETALHADO AS
SELECT 
    s.COD_IBGE_MUNICIPIO,
    i.NOME_MUNICIPIO,
    i.NOME_MICRORREGIAO,
    i.NOME_MESORREGIAO,
    YEAR(s.DATA_NOTIFICACAO) AS ano,
    COUNT(*) AS total_casos,
    SUM(s.FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(s.FLAG_OBITO_DENGUE) AS obitos,
    ROUND(SUM(s.FLAG_HOSPITALIZACAO) * 100.0 / COUNT(*), 2) AS taxa_hospitalizacao_pct,
    ROUND(SUM(s.FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 2) AS taxa_mortalidade_pct,
    MIN(s.DATA_NOTIFICACAO) AS primeira_notificacao,
    MAX(s.DATA_NOTIFICACAO) AS ultima_notificacao
FROM fiap.silver.SINAN_DENGUE_SP s
LEFT JOIN fiap.silver.IBGE_SP i ON CAST(s.COD_IBGE_MUNICIPIO AS BIGINT) = CAST(i.COD_SUS AS BIGINT)
GROUP BY s.COD_IBGE_MUNICIPIO, i.NOME_MUNICIPIO, i.NOME_MICRORREGIAO, i.NOME_MESORREGIAO, YEAR(s.DATA_NOTIFICACAO)
""")
print("tabela SINAN_DENGUE_MUNICIPIOS_DETALHADO criada")

# tabela por faixa etaria
print("\n criando tabela SINAN_DENGUE_FAIXA_ETARIA")
spark.sql("""
CREATE OR REPLACE TABLE fiap.gold.SINAN_DENGUE_FAIXA_ETARIA AS
SELECT 
    CASE 
        WHEN IDADE_ANOS < 5 THEN '0-4 anos'
        WHEN IDADE_ANOS < 15 THEN '5-14 anos'
        WHEN IDADE_ANOS < 30 THEN '15-29 anos'
        WHEN IDADE_ANOS < 45 THEN '30-44 anos'
        WHEN IDADE_ANOS < 60 THEN '45-59 anos'
        WHEN IDADE_ANOS >= 60 THEN '60+ anos'
        ELSE 'Desconhecido'
    END AS faixa_etaria,
    YEAR(DATA_NOTIFICACAO) AS ano,
    GENERO,
    COUNT(*) AS total_casos,
    SUM(FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(FLAG_OBITO_DENGUE) AS obitos,
    ROUND(SUM(FLAG_HOSPITALIZACAO) * 100.0 / COUNT(*), 2) AS taxa_hospitalizacao_pct,
    ROUND(SUM(FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 2) AS taxa_mortalidade_pct
FROM fiap.silver.SINAN_DENGUE_SP
WHERE IDADE_ANOS IS NOT NULL
GROUP BY faixa_etaria, YEAR(DATA_NOTIFICACAO), GENERO
""")
print("tabela SINAN_DENGUE_FAIXA_ETARIA criada")

# 7.3: Tabela de resumo mensal por indicador
print("\ncriando tabela SINAN_DENGUE_RESUMO_MENSAL")
spark.sql("""
CREATE OR REPLACE TABLE fiap.gold.SINAN_DENGUE_RESUMO_MENSAL AS
SELECT 
    CAST(DATE_TRUNC('month', DATA_NOTIFICACAO) AS DATE) AS mes,
    YEAR(DATA_NOTIFICACAO) AS ano,
    MONTH(DATA_NOTIFICACAO) AS mes_num,
    COUNT(*) AS total_casos,
    SUM(FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(FLAG_OBITO_DENGUE) AS obitos,
    COUNT(DISTINCT COD_IBGE_MUNICIPIO) AS municipios_afetados,
    ROUND(SUM(FLAG_HOSPITALIZACAO) * 100.0 / COUNT(*), 2) AS taxa_hospitalizacao_pct,
    ROUND(SUM(FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 2) AS taxa_mortalidade_pct
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY mes, ano, mes_num
ORDER BY mes
""")
print("tabela SINAN_DENGUE_RESUMO_MENSAL criada")

# 7.4: Tabela de indicadores por sexo e idade
print("\ncriando tabela SINAN_DENGUE_INDICADORES")
spark.sql("""
CREATE OR REPLACE TABLE fiap.gold.SINAN_DENGUE_INDICADORES AS
SELECT 
    YEAR(DATA_NOTIFICACAO) AS ano,
    MONTH(DATA_NOTIFICACAO) AS mes,
    GENERO,
    CASE 
        WHEN IDADE_ANOS < 15 THEN 'Criança (0-14)'
        WHEN IDADE_ANOS < 60 THEN 'Adulto (15-59)'
        WHEN IDADE_ANOS >= 60 THEN 'Idoso (60+)'
        ELSE 'Desconhecido'
    END AS grupo_etario,
    COUNT(*) AS total_casos,
    SUM(FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(FLAG_OBITO_DENGUE) AS obitos,
    ROUND(SUM(FLAG_HOSPITALIZACAO) * 100.0 / COUNT(*), 2) AS taxa_hosp_pct,
    ROUND(SUM(FLAG_OBITO_DENGUE) * 100.0 / COUNT(*), 2) AS taxa_morte_pct
FROM fiap.silver.SINAN_DENGUE_SP
WHERE IDADE_ANOS IS NOT NULL
GROUP BY ano, mes, GENERO, grupo_etario
""")
print("tabela SINAN_DENGUE_INDICADORES criada")

In [0]:
df_municipios_det = spark.table("fiap.gold.SINAN_DENGUE_MUNICIPIOS_DETALHADO")
enviar_para_supabase(df_municipios_det,"sinan_dengue_municipios_detalhado"
)

df_faixa_det = spark.table("fiap.gold.SINAN_DENGUE_FAIXA_ETARIA")
enviar_para_supabase(df_faixa_det, "sinan_dengue_faixa_etaria")

df_resumo_mensal = spark.table("fiap.gold.sinan_dengue_resumo_mensal")
enviar_para_supabase(df_resumo_mensal, "sinan_dengue_resumo_mensal")

df_indicadores = spark.table("fiap.gold.SINAN_DENGUE_INDICADORES")
enviar_para_supabase(df_indicadores, "sinan_dengue_indicadores")

##visualizações

In [0]:

print("\n visualizações")

# grafico de linha com preenchimento para enxergar como o volume de casos subiu ou desceu ao longo dos dias
print("\ngerando tendencia temporal")
df_tendencia = spark.sql("""
SELECT 
    CAST(DATA_NOTIFICACAO AS DATE) AS data,
    COUNT(*) AS casos
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY CAST(DATA_NOTIFICACAO AS DATE)
ORDER BY data
""").toPandas()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_tendencia['data'], df_tendencia['casos'], linewidth=2, color='#1f77b4')
ax.fill_between(df_tendencia['data'], df_tendencia['casos'], alpha=0.3, color='#1f77b4')
ax.set_xlabel('Data', fontsize=12, fontweight='bold')
ax.set_ylabel('Número de Casos', fontsize=12, fontweight='bold')
ax.set_title('Tendencia temporal de casos de dengue - SP (2022-2025)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()


# comparando casos por genero
print("\ngerando: distri por sexo")
df_sexo = spark.sql("""
SELECT GENERO, COUNT(*) AS casos
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY GENERO
ORDER BY casos DESC
""").toPandas()

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(df_sexo['GENERO'], df_sexo['casos'], color=['#ff7f0e', '#2ca02c', '#d62728'])
ax.set_xlabel('Numero de casos', fontsize=12, fontweight='bold')
ax.set_title('Distribuicao de casos por sexo', fontsize=14, fontweight='bold')
for i, v in enumerate(df_sexo['casos']):
    ax.text(v, i, f' {v:,.0f}', va='center', fontweight='bold')
plt.tight_layout()


# entender quais as idades que mais tem casos
print("\ngerando distribuição por faixa etaria")
df_idade = spark.sql("""
SELECT 
    CASE 
        WHEN IDADE_ANOS < 5 THEN '0-4'
        WHEN IDADE_ANOS < 15 THEN '5-14'
        WHEN IDADE_ANOS < 30 THEN '15-29'
        WHEN IDADE_ANOS < 45 THEN '30-44'
        WHEN IDADE_ANOS < 60 THEN '45-59'
        WHEN IDADE_ANOS >= 60 THEN '60+'
        ELSE 'Desc'
    END AS faixa,
    COUNT(*) AS casos
FROM fiap.silver.SINAN_DENGUE_SP
WHERE IDADE_ANOS IS NOT NULL
GROUP BY faixa
ORDER BY 
    CASE 
        WHEN faixa = '0-4' THEN 1
        WHEN faixa = '5-14' THEN 2
        WHEN faixa = '15-29' THEN 3
        WHEN faixa = '30-44' THEN 4
        WHEN faixa = '45-59' THEN 5
        WHEN faixa = '60+' THEN 6
        ELSE 7
    END
""").toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(df_idade['faixa'], df_idade['casos'], color='#1f77b4', edgecolor='black', linewidth=1.5)
ax.set_xlabel('Faixa etaria', fontsize=12, fontweight='bold')
ax.set_ylabel('Numero de casos', fontsize=12, fontweight='bold')
ax.set_title('Distribuiçao de Casos por faixa etaria', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}k' if x >= 1000 else f'{int(x)}'))
for i, v in enumerate(df_idade['casos']):
    ax.text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()


# analise do comportamento da doença mes a mes para ver os periodos de pico sazonais
print("\ngerando sazonalidade")
df_sazon = spark.sql("""
SELECT 
    MONTH(DATA_NOTIFICACAO) AS mes,
    COUNT(*) AS casos
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY MONTH(DATA_NOTIFICACAO)
ORDER BY mes
""").toPandas()

meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(1, 13), df_sazon['casos'], color='#2ca02c', edgecolor='black', linewidth=1.5)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses)
ax.set_xlabel('Mês', fontsize=12, fontweight='bold')
ax.set_ylabel('Número de Casos', fontsize=12, fontweight='bold')
ax.set_title('Sazonalidade da dengue - padrão medio (2020-2026)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}k' if x >= 1000 else f'{int(x)}'))
plt.tight_layout()


# Avaliando  gravidade por taxas de internação e mortalidade comparadas ano a ano
print("\ngerando desfechos por ano")
df_desfechos = spark.sql("""
SELECT 
    ANO_NOTIFIC AS ano,
    COUNT(*) AS total,
    SUM(FLAG_HOSPITALIZACAO) AS hosp,
    SUM(FLAG_OBITO_DENGUE) AS obitos
FROM fiap.silver.SINAN_DENGUE_SP
GROUP BY ANO_NOTIFIC
ORDER BY ANO_NOTIFIC
""").toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# casos totais por ano
ax1.bar(df_desfechos['ano'].astype(str), df_desfechos['total'], color='#1f77b4', edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Ano', fontsize=12, fontweight='bold')
ax1.set_ylabel('Número de casos', fontsize=12, fontweight='bold')
ax1.set_title('Total de casos por ano', fontsize=14, fontweight='bold')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}k' if x >= 1000 else f'{int(x)}'))
for i, v in enumerate(df_desfechos['total']):
    ax1.text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# transofmrnado os números brutos em porcentagem
df_desfechos['taxa_hosp'] = (df_desfechos['hosp'] / df_desfechos['total'] * 100).round(2)
df_desfechos['taxa_morte'] = (df_desfechos['obitos'] / df_desfechos['total'] * 100).round(2)

x = np.arange(len(df_desfechos))
width = 0.35
ax2.bar(x - width/2, df_desfechos['taxa_hosp'], width, label='Taxa hospitalizaçao (%)', color='#ff7f0e')
ax2.bar(x + width/2, df_desfechos['taxa_morte'], width, label='Taxa mortalidade (%)', color='#d62728')
ax2.set_xlabel('ano', fontsize=12, fontweight='bold')
ax2.set_ylabel('taxa (%)', fontsize=12, fontweight='bold')
ax2.set_title('Taxas de hospitalizaçao e mortalidade por ano', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(df_desfechos['ano'].astype(str))
ax2.legend()

plt.tight_layout()


# 10 municipios com mais casos
print("\ngerando top 10 municípios")
df_municipios = spark.sql("""
SELECT 
    i.NOME_MUNICIPIO,
    COUNT(*) AS casos
FROM fiap.silver.SINAN_DENGUE_SP s
LEFT JOIN fiap.silver.IBGE_SP i ON CAST(s.COD_IBGE_MUNICIPIO AS BIGINT) = CAST(i.COD_SUS AS BIGINT)
GROUP BY i.NOME_MUNICIPIO
ORDER BY casos DESC
LIMIT 10
""").toPandas()

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(df_municipios['NOME_MUNICIPIO'], df_municipios['casos'], color='#d62728', edgecolor='black', linewidth=1.5)
ax.set_xlabel('Número de casos', fontsize=12, fontweight='bold')
ax.set_title('Top 10 municípios com mais casos de dengue - SP', fontsize=14, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(df_municipios['casos']):
    ax.text(v, i, f' {v:,.0f}', va='center', fontweight='bold')
plt.tight_layout()